# Circuit Diagram Explanation Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B that explains electronic circuits in plain language,
generates SPICE netlists, and answers what-if questions. Designed for students
in low-resource settings who lack access to expensive simulation software.

Runs fully offline on consumer hardware (T4 GPU). No cloud dependency.

In [ ]:
# Install dependencies
!pip install unsloth[colab-new] trl datasets -q
!git clone https://github.com/govindrathore27/gemma4-engineering-diagrams.git /kaggle/working/repo 2>/dev/null || git -C /kaggle/working/repo pull
import sys
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Train the Circuit Tutor model
import os
os.environ["WANDB_DISABLED"] = "true"

from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="circuit",
    data_path="/kaggle/input/gemma4-circuit-train-data/train.jsonl",
    output_dir="/kaggle/working/circuit_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Evaluate on held-out eval set
import json
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/circuit_adapter")
print("Model loaded")

eval_rows = [json.loads(l) for l in open("/kaggle/input/gemma4-circuit-train-data/eval.jsonl", encoding="utf-8")]
eval_rows = eval_rows[:100]

predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected = [r["output"] for r in eval_rows]

bleu = evaluate_batch(predictions, expected, metric="bleu")
f1 = evaluate_batch(predictions, expected, metric="f1")
print(f"BLEU-1: {bleu['mean']:.3f}")
print(f"Token F1: {f1['mean']:.3f}")

with open("/kaggle/working/eval_results.txt", "w") as f:
    f.write(f"BLEU-1: {bleu['mean']:.3f}\n")
    f.write(f"Token F1: {f1['mean']:.3f}\n")
    f.write(f"Evaluated on {len(eval_rows)} samples\n")

In [ ]:
# Export to GGUF for offline deployment
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/circuit_adapter",
    output_path="/kaggle/working/circuit_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Demo inference - Circuit Explanation Tutor
sample_circuit = """
Components: 2x resistor, 1x capacitor, 1x diode
Connections: R1 in series with D1; R2 and C1 in parallel across D1
"""

questions = [
    "Describe what this circuit does based on its components.",
    "How many components are in this circuit?",
    "What limits the current through the diode in this circuit?",
    "Generate a SPICE-compatible netlist for this circuit.",
    "If all resistors were doubled in value, how would that affect the circuit?",
]

print("=== Circuit Diagram Explanation Tutor Demo ===\n")
for q in questions:
    answer = run(model, tokenizer, q, sample_circuit)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## Real-World Impact

In underserved communities, students learning electronics often have textbooks but no
access to expensive tools like LTspice or Multisim. A photo of a hand-drawn notebook
circuit can now be explained in plain language and converted to a simulation-ready netlist.

**Key capabilities:**
- Explains circuit function from component list
- Identifies current-limiting and protective components
- Generates SPICE-compatible netlists for simulation
- Answers what-if questions about component value changes

**Deployment:** Runs fully offline on a T4 GPU (15 GB VRAM). GGUF export enables
CPU-only inference via llama.cpp — suitable for schools with no reliable internet.

**Datasets used:**
- circuit1k: 1000 annotated circuit images (YOLO format)
- Training: 3,484 auto-generated QA pairs from component annotations